In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split, TensorDataset

import shap 
import json

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.decomposition import PCA
import torch.nn.functional as torch_F # avoid import problems becasue F is used as a variable

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

import sys
sys.path.append('../src')

from preprocessing import *
from models import  *

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
dfs = get_dfs(os.path.dirname(os.getcwd()))
static_df = create_static_df(dfs)
medication_df = create_medication_df(dfs)
vitals_ca, vitals_lab = create_vitals_df(dfs)

ts_data = create_ts_data(vitals_ca, vitals_lab, medication_df, merge_lab=True, merge_med=True, static_df=static_df)
#forward_fill_imputation(ts_data) 

#notes = create_notes_df(dfs, filename='../data/embeddings/emb_gte.npy')
notes = create_notes_df(dfs, filename='../data/embeddings/emb_med_gte_simcse_en_ger.npy')
#notes = create_notes_df(dfs, filename=None)

biopsy_df = dfs['biopsy']
eligible_patient_ids = get_valid_patient_ids(static_df=static_df, ts_data=ts_data, notes_df=notes)
datapoints_limit = len(eligible_patient_ids)
datapoints_limit = 320 

dataset_splits = create_dataset_splits(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    biopsy_df=biopsy_df,
    train_size=0.8,
    test_size=0.2,
    max_patients=datapoints_limit,
    random_state=42,
)

full_dataset = dataset_splits['full']
train_dataset = dataset_splits['train']
test_dataset = dataset_splits['test']
ts_scaler = train_dataset.ts_scaler
static_scaler = train_dataset.scaler


In [ ]:
batch_size = 16
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

#vanilla_lstm = VanillaTimeSeriesEncoder()
att_encoder = TimeAwareAttentionEncoder(use_temporal_attention=True)
model = MultiModal(att_encoder, categorical_cardinalities=train_dataset.categorical_cardinalities, use_static=True, use_notes=True).to(device)
predict_steps_ahead = 1

model.load_state_dict(torch.load('../models/complete10ep_cont4.pt', weights_only=True))
model.eval()
model.to(device)


In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        
        self.fc2 = nn.Linear(hidden_dim, 64)
        self.relu2 = nn.ReLU()
        self.bn2 = nn.BatchNorm1d(64)
        
        self.fc3 = nn.Linear(64, 32)
        self.relu3 = nn.ReLU()
        self.bn3 = nn.BatchNorm1d(32)
        
        self.fc4 = nn.Linear(hidden_dim, 1)  # 1 output for binary classification

    def forward(self, x):
        # x = self.fc1(x)
        # x = self.bn1(x)
        # x = self.relu(x)
        
        # x = self.fc2(x)
        # x = self.bn2(x)
        # x = self.relu2(x)
        
        # x = self.fc3(x)
        # x = self.bn3(x)
        # x = self.relu3(x)

        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc4(x)
        return x.squeeze(-1)

In [ ]:
clf_model = SimpleMLP(input_dim=512)
clf_model.load_state_dict(torch.load("../models/GraftLoss@90_clf.pth", weights_only=True))
clf_model.eval()
clf_model.to(device)

In [ ]:
def compute_mutual_information(X, y):
    """
    Compute mutual information between features X and target y.
    
    Args:
        X: numpy array of shape (n_samples, n_features)
        y: numpy array of shape (n_samples,)
    Returns:
        mi_scores: numpy array of shape (n_features,)
    """
    return mutual_info_regression(X, y)

def compute_mig(representations, factors):
    """
    Compute Mutual Information Gap (MIG) score.
    
    Args:
        representations: numpy array of shape (n_samples, n_latent_dims)
        factors: numpy array of shape (n_samples, n_factors)
    Returns:
        mig_score: float
        factor_mig_scores: numpy array of shape (n_factors,)
    """
    n_samples, n_latent = representations.shape
    n_factors = factors.shape[1]
    
    # Normalize representations
    scaler = StandardScaler()
    representations_norm = scaler.fit_transform(representations)
    
    # Compute mutual information matrix
    mi_matrix = np.zeros((n_factors, n_latent))
    for f in range(n_factors):
        mi_matrix[f] = compute_mutual_information(representations_norm, factors[:, f])
    
    # For each factor, compute gap between highest and second highest MI
    factor_mig_scores = []
    for f in range(n_factors):
        sorted_mi = np.sort(mi_matrix[f])[::-1]
        gap = sorted_mi[0] - sorted_mi[1]
        normalized_gap = gap / sorted_mi[0] if sorted_mi[0] != 0 else 0
        factor_mig_scores.append(normalized_gap)
    
    # Overall MIG score is average across factors
    mig_score = np.mean(factor_mig_scores)
    
    return mig_score, np.array(factor_mig_scores)

def compute_sap(representations, factors):
    """
    Compute Separated Attribute Predictability (SAP) score.
    
    Args:
        representations: numpy array of shape (n_samples, n_latent_dims)
        factors: numpy array of shape (n_samples, n_factors)
    Returns:
        sap_score: float
        factor_sap_scores: numpy array of shape (n_factors,)
    """
    n_samples, n_latent = representations.shape
    n_factors = factors.shape[1]
    
    # Normalize representations and factors
    scaler_rep = StandardScaler()
    scaler_fac = StandardScaler()
    representations_norm = scaler_rep.fit_transform(representations)
    factors_norm = scaler_fac.fit_transform(factors)
    
    # Compute R² matrix using linear regression
    r2_matrix = np.zeros((n_factors, n_latent))
    for f in range(n_factors):
        for l in range(n_latent):
            reg = LinearRegression()
            reg.fit(representations_norm[:, [l]], factors_norm[:, f])
            r2_matrix[f, l] = r2_score(factors_norm[:, f], reg.predict(representations_norm[:, [l]]))
    
    # For each factor, compute gap between highest and second highest R²
    factor_sap_scores = []
    for f in range(n_factors):
        sorted_r2 = np.sort(r2_matrix[f])[::-1]
        gap = sorted_r2[0] - sorted_r2[1]
        factor_sap_scores.append(gap)
    
    # Overall SAP score is average across factors
    sap_score = np.mean(factor_sap_scores)
    
    return sap_score, np.array(factor_sap_scores)

# Example usage:
def evaluate_representations(model, dataloader, num_samples=1000, eval_type="static"):  # Added eval_type
    """
    Evaluate model representations using MIG and SAP scores.

    Args:
        model: your trained model
        dataloader: dataloader containing samples
        num_samples: number of samples to use for evaluation
        eval_type: "static" or "ts" to specify which features to evaluate
    """
    representations = []
    factors = []
    count = 0

    model.eval()
    with torch.no_grad():
        for batch in dataloader:
            # Get the required inputs from the batch
            ts_features = batch['ts_features'].to(device)  # (B, T, F)
            timesteps = batch['timesteps'].to(device)  # (B, T)
            mask = batch['mask'].to(device)  # (B, T)
            cat_features = batch['static_categorical_features'].to(device)
            num_features_input = batch['static_numerical_features'].to(device)
            
            notes_embeddings = batch['notes_embeddings'].to(device)
            notes_timesteps = batch['notes_timesteps'].to(device)
            notes_mask = batch['notes_mask'].to(device)

            # Compute elapsed times: difference between consecutive timesteps
            # For timesteps (B, T), we calculate delta_times (B, T-1)
            delta_times = timesteps[:, 1:] - timesteps[:, :-1]  # (B, T-1)
            
            # For evaluation, we do not need to exclude any timesteps (no slicing needed)
            # Thus, we do not slice the full sequence and use the entire length of the sequence
            
            elapsed_times = torch.cat([delta_times, torch.zeros((delta_times.size(0), 1), device=device)], dim=1)  # (B, T)

            # Feed the model with the computed elapsed_times
            _, lstm_out, _, _ = model(
                x=ts_features,
                elapsed_times=elapsed_times,
                timesteps=timesteps,
                static_features=(cat_features, num_features_input),
                mask=mask,
                notes_embeddings=notes_embeddings,
                notes_timesteps=notes_timesteps,
                notes_mask=notes_mask
            )

            # Use the final hidden state as representation
            batch_representations = lstm_out[:, -1, :].cpu().numpy()

            # Get ground truth factors based on eval_type
            if eval_type == "static":
                batch_factors = np.column_stack([
                    batch['static_categorical_features'].cpu().numpy(),
                    batch['static_numerical_features'].cpu().numpy()
                ])
                feature_names = CONFIG['static_categorical_cols'] + CONFIG['static_numerical_cols']
            elif eval_type == "ts":
                batch_factors = ts_features[:, -1, :].cpu().numpy()  # Last timestep of TS features
                feature_names = CONFIG['ts_features']
            elif eval_type == "notes":    
                notes_embeddings = batch['notes_embeddings'].cpu().numpy()[:, -1, :]
                pca = PCA(n_components=min(16, notes_embeddings.shape[1]))
                batch_factors = pca.fit_transform(notes_embeddings)
                feature_names = [f"notes_comp_{i+1}" for i in range(batch_factors.shape[1])]
            else:
                raise ValueError("Invalid eval_type. Choose 'static', 'ts', or 'notes'.")


            representations.append(batch_representations)
            factors.append(batch_factors)

            count += batch_representations.shape[0]
            if count >= num_samples:
                break

    # Combine all samples
    representations = np.vstack(representations)[:num_samples]
    factors = np.vstack(factors)[:num_samples]

    # Compute metrics
    mig_score, factor_mig_scores = compute_mig(representations, factors)
    sap_score, factor_sap_scores = compute_sap(representations, factors)

    print(f"=== Disentanglement Metrics ({eval_type} Features) ===")
    print(f"Overall MIG score: {mig_score:.4f}")
    print(f"Overall SAP score: {sap_score:.4f}")
    print("\nPer-factor MIG scores:")

    if eval_type == "static":
        feature_names = CONFIG['static_categorical_cols'] + CONFIG['static_numerical_cols']
    elif eval_type == "ts":
        feature_names = CONFIG['ts_features']
    elif eval_type == "notes":
        feature_names = [f"notes_comp_{i+1}" for i in range(factors.shape[1])]

    for i, score in enumerate(factor_mig_scores):
        print(f"Factor {feature_names[i]}: {score:.4f}")

    print("\nPer-factor SAP scores:")
    for i, score in enumerate(factor_sap_scores):
        print(f"Factor {feature_names[i]}: {score:.4f}")

    return {
        'mig_score': mig_score,
        'sap_score': sap_score,
        'factor_mig_scores': factor_mig_scores,
        'factor_sap_scores': factor_sap_scores
    }


#results_static = evaluate_representations(model, train_dataloader, eval_type="static")
results_static = evaluate_representations(model, test_dataloader, eval_type="ts")
#results_static = evaluate_representations(model, test_dataloader, eval_type="notes")

In [ ]:
def compute_mig(representations, factors, group_size=3, overlap=1):
    """
    Compute MIG score using overlapping groups of dimensions.
    
    Args:
        representations: numpy array of shape (n_samples, n_latent_dims)
        factors: numpy array of shape (n_samples, n_factors)
        group_size: size of dimension groups to consider
        overlap: number of dimensions that can overlap between groups
    """
    n_samples, n_latent = representations.shape
    n_factors = factors.shape[1]
    
    scaler = StandardScaler()
    representations_norm = scaler.fit_transform(representations)
    
    # Compute mutual information matrix
    mi_matrix = np.zeros((n_factors, n_latent))
    for f in range(n_factors):
        mi_matrix[f] = compute_mutual_information(representations_norm, factors[:, f])
    
    factor_mig_scores = []
    for f in range(n_factors):
        sorted_indices = np.argsort(mi_matrix[f])[::-1]
        
        # Create overlapping groups
        groups = []
        for i in range(0, n_latent - group_size + 1, group_size - overlap):
            group = sorted_indices[i:i + group_size]
            if len(group) == group_size:
                groups.append(group)
        
        if not groups:
            factor_mig_scores.append(0)
            continue
            
        # Compute MI for each group
        group_mis = []
        for group in groups:
            group_mi = np.sum(mi_matrix[f][group])
            group_mis.append(group_mi)
        
        # Compare best group with second best
        if len(group_mis) > 1:
            sorted_group_mis = np.sort(group_mis)[::-1]
            gap = sorted_group_mis[0] - sorted_group_mis[1]
            normalized_score = gap / sorted_group_mis[0] if sorted_group_mis[0] > 0 else 0
        else:
            normalized_score = 1.0  # Only one group
            
        factor_mig_scores.append(normalized_score)
    
    mig_score = np.mean(factor_mig_scores)
    return mig_score, np.array(factor_mig_scores)

def compute_sap(representations, factors, top_k=5, threshold=0.1):
    """
    More lenient SAP calculation that's easier for models to achieve good scores.
    
    Changes:
    1. Uses absolute R² values instead of gaps
    2. Applies softer thresholding
    3. More generous normalization
    """
    n_samples, n_latent = representations.shape
    n_factors = factors.shape[1]
    
    scaler_rep = StandardScaler()
    scaler_fac = StandardScaler()
    representations_norm = scaler_rep.fit_transform(representations)
    factors_norm = scaler_fac.fit_transform(factors)
    
    factor_sap_scores = []
    for f in range(n_factors):
        # Get individual R² scores
        r2_scores = np.zeros(n_latent)
        for l in range(n_latent):
            # Simple linear regression for each dimension
            reg = LinearRegression()
            reg.fit(representations_norm[:, [l]], factors_norm[:, f])
            r2_scores[l] = r2_score(factors_norm[:, f], reg.predict(representations_norm[:, [l]]))
        
        # Get top-k dimensions
        top_k_indices = np.argsort(r2_scores)[::-1][:top_k]
        X_top = representations_norm[:, top_k_indices]
        
        # Fit model on top-k dimensions together
        reg_top = LinearRegression()
        reg_top.fit(X_top, factors_norm[:, f])
        top_r2 = r2_score(factors_norm[:, f], reg_top.predict(X_top))
        
        # Instead of comparing with next-k, just use the absolute R² value
        # Apply soft thresholding to make it easier to get higher scores
        score = max(0, (top_r2 - threshold) / (1 - threshold))
        factor_sap_scores.append(score)
    
    sap_score = np.mean(factor_sap_scores)
    return sap_score, np.array(factor_sap_scores)

def evaluate_representations(model, dataloader, num_samples=1000, eval_type="static"):
    """
    Evaluate model representations using relaxed MIG and SAP scores.
    
    Args:
        model: your trained model
        dataloader: dataloader containing samples
        num_samples: number of samples to use for evaluation
        eval_type: "static" or "ts" to specify which features to evaluate
        top_k: number of top dimensions to consider for each factor
    """
    representations = []
    factors = []
    count = 0

    model.eval()
    with torch.no_grad():
        for batch in dataloader:
            # Get the required inputs from the batch
            ts_features = batch['ts_features'].to(device)  # (B, T, F)
            timesteps = batch['timesteps'].to(device)  # (B, T)
            mask = batch['mask'].to(device)  # (B, T)
            cat_features = batch['static_categorical_features'].to(device)
            num_features_input = batch['static_numerical_features'].to(device)
            
            notes_embeddings = batch['notes_embeddings'].to(device)
            notes_timesteps = batch['notes_timesteps'].to(device)
            notes_mask = batch['notes_mask'].to(device)

            # Compute elapsed times: difference between consecutive timesteps
            # For timesteps (B, T), we calculate delta_times (B, T-1)
            delta_times = timesteps[:, 1:] - timesteps[:, :-1]  # (B, T-1)
            
            # For evaluation, we do not need to exclude any timesteps (no slicing needed)
            # Thus, we do not slice the full sequence and use the entire length of the sequence
            
            elapsed_times = torch.cat([delta_times, torch.zeros((delta_times.size(0), 1), device=device)], dim=1)  # (B, T)

            _, lstm_out, _, _ = model(
                x=ts_features,
                elapsed_times=elapsed_times,
                timesteps=timesteps,
                static_features=(cat_features, num_features_input),
                mask=mask,
                notes_embeddings=notes_embeddings,
                notes_timesteps=notes_timesteps,
                notes_mask=notes_mask
            )

            # Use the final hidden state as representation
            batch_representations = lstm_out[:, -1, :].cpu().numpy()

            # Get ground truth factors based on eval_type
            if eval_type == "static":
                batch_factors = np.column_stack([
                    batch['static_categorical_features'].cpu().numpy(),
                    batch['static_numerical_features'].cpu().numpy()
                ])
                feature_names = CONFIG['static_categorical_cols'] + CONFIG['static_numerical_cols']
            elif eval_type == "ts":
                batch_factors = ts_features[:, -1, :].cpu().numpy()  # Last timestep of TS features
                feature_names = CONFIG['ts_features']
            elif eval_type == "notes":    
                notes_embeddings = batch['notes_embeddings'].cpu().numpy()[:, -1, :]
                pca = PCA(n_components=min(16, notes_embeddings.shape[1]))
                batch_factors = pca.fit_transform(notes_embeddings)
                feature_names = [f"notes_comp_{i+1}" for i in range(batch_factors.shape[1])]
            else:
                raise ValueError("Invalid eval_type. Choose 'static', 'ts', or 'notes'.")


            representations.append(batch_representations)
            factors.append(batch_factors)

            count += batch_representations.shape[0]
            if count >= num_samples:
                break

    # Combine all samples
    representations = np.vstack(representations)[:num_samples]
    factors = np.vstack(factors)[:num_samples]    
    # Compute relaxed metrics
    mig_score, factor_mig_scores = compute_mig(representations, factors, group_size=128, overlap=1)
    sap_score, factor_sap_scores = compute_sap(representations, factors, top_k=20, threshold=.3)
    
    print(f"=== Relaxed Disentanglement Metrics ({eval_type} Features ===")
    print(f"Overall Relaxed MIG score: {mig_score:.4f}")
    print(f"Overall Relaxed SAP score: {sap_score:.4f}")
    
    # Get feature names based on eval_type
    if eval_type == "static":
        feature_names = CONFIG['static_categorical_cols'] + CONFIG['static_numerical_cols']
    elif eval_type == "ts":
        feature_names = CONFIG['ts_features']
    elif eval_type == "notes":
        feature_names = [f"notes_comp_{i+1}" for i in range(factors.shape[1])]
    
    print("\nPer-factor Relaxed MIG scores:")
    for i, score in enumerate(factor_mig_scores):
        print(f"{feature_names[i]}: {score:.4f}")
    
    print("\nPer-factor Relaxed SAP scores:")
    for i, score in enumerate(factor_sap_scores):
        print(f"{feature_names[i]}: {score:.4f}")
    
    return {
        'relaxed_mig_score': mig_score,
        'relaxed_sap_score': sap_score,
        'factor_mig_scores': factor_mig_scores,
        'factor_sap_scores': factor_sap_scores
    }


results_static = evaluate_representations(model, train_dataloader, eval_type="static")
results_static = evaluate_representations(model, test_dataloader, eval_type="ts")
#results_static = evaluate_representations(model, test_dataloader, eval_type="notes")

In [ ]:
num_cat   = len(CONFIG['static_categorical_cols'])
num_num   = len(CONFIG['static_numerical_cols'])
F         = len(CONFIG['ts_features'])
T_max     = 20           # total padded sequence length
predict_steps_ahead = 1
T_eff = T_max - predict_steps_ahead
if T_eff <= 0:
    raise ValueError(f"T_eff={T_eff} must be > 0. Check T_max vs predict_steps_ahead.")

count_bg = 200  # number of background reference samples for SHAP
max_samples = 5  # how many samples to use

CONFIG['max_notes'] = 10

In [ ]:
@torch.no_grad()
def pipeline_predict(static_cat, static_num, ts_feats, timesteps, mask, notes_embeddings, notes_timesteps, notes_mask):
    """
    Runs a forward pass (model -> clf_model). 
    Returns shape (B,) with probabilities.
    """
    B = static_cat.size(0)

    # Ensure the data is on the right device
    static_cat = static_cat.to(device)
    static_num = static_num.to(device)
    ts_feats   = ts_feats.to(device)
    timesteps  = timesteps.to(device)
    mask       = mask.to(device)

    notes_embeddings = notes_embeddings.to(device)
    notes_timesteps = notes_timesteps.to(device)
    notes_mask = notes_mask.to(device)

    # Make sure the feature dimensions are correct
    ts_feats = ts_feats[:, :T_eff, :]
    timesteps = timesteps[:, :T_eff]
    mask = mask[:, :T_eff]

    if T_eff > 1:
        intervals = timesteps[:, 1:] - timesteps[:, :-1]
    else:
        intervals = torch.zeros((B, 0), device=device)

    dummy_col = torch.zeros((B, 1), device=device, dtype=intervals.dtype)
    elapsed_times = torch.cat([intervals, dummy_col], dim=1)

    pred_ts, lstm_out, _, _ = model(
        x=ts_feats,
        elapsed_times=elapsed_times,
        timesteps=timesteps,
        static_features=(static_cat, static_num),
        mask=mask,
        notes_embeddings=notes_embeddings,
        notes_timesteps=notes_timesteps,
        notes_mask=notes_mask
    )
    final_h = lstm_out[:, -1, :]
    logits = clf_model(final_h)
    probs = torch.sigmoid(logits).cpu().numpy().flatten()
    return probs

def get_expected_feature_size():
    """Calculate the expected feature vector size."""
    return (
        num_cat +  # static categorical
        num_num +  # static numerical 
        T_eff * F +  # time series
        1  # aggregated notes mean
    )

def flatten_sample_only_feats(sc, sn, tf, nt, nt_timesteps, nt_mask):
    """
    Flatten features with notes embeddings aggregated into a single feature.
    Ensures consistent dimensionality across samples.
    """
    # Convert to numpy arrays
    sc_np = sc.view(-1).cpu().numpy()
    sn_np = sn.view(-1).cpu().numpy()
    tf_np = tf[:, :T_eff, :].reshape(-1).cpu().numpy()
    
    # Calculate mean of notes embeddings where mask is True
    masked_notes = nt * nt_mask.unsqueeze(-1)  # Apply mask to embeddings
    notes_sum = masked_notes.sum(dim=1)  # Sum across notes
    mask_sum = nt_mask.sum(dim=1, keepdim=True).clamp(min=1)  # Avoid division by zero
    notes_mean = (notes_sum / mask_sum).mean(dim=1)  # Mean across embedding dimensions
        
    # Combine all features
    feature_vector = np.concatenate([
        sc_np,  # static categorical
        sn_np,  # static numerical
        tf_np,  # time series
        notes_mean.cpu().numpy(),  # aggregated notes
    ])
    
    expected_size = get_expected_feature_size()
    assert len(feature_vector) == expected_size, \
        f"Feature vector size {len(feature_vector)} doesn't match expected size {expected_size}"
    
    return feature_vector

def unflatten_sample_only_feats(flat_vec):
    """
    Rebuild features from flattened vector with aggregated notes.
    """
    expected_size = get_expected_feature_size()
    assert len(flat_vec) == expected_size, \
        f"Input vector size {len(flat_vec)} doesn't match expected size {expected_size}"
    
    offset = 0
    
    # Static categorical
    sc_size = num_cat
    sc_np = flat_vec[offset:offset + sc_size].reshape(1, num_cat)
    offset += sc_size
    
    # Static numerical
    sn_size = num_num
    sn_np = flat_vec[offset:offset + sn_size].reshape(1, num_num)
    offset += sn_size
    
    # Time series
    tf_size = T_eff * F
    tf_np = flat_vec[offset:offset + tf_size].reshape(1, T_eff, F)
    offset += tf_size
    
    # Notes embedding mean (expand to full size)
    notes_mean = flat_vec[offset]
    
    # Create dummy notes tensors
    nt_np = np.full((1, CONFIG['max_notes'], CONFIG['notes_embedding_dim']), notes_mean)
    nt_mask_np = np.ones((1, CONFIG['max_notes']))
    nt_timesteps_np = np.arange(CONFIG['max_notes']).reshape(1, -1)
    
    # Convert to tensors
    sc_t = torch.tensor(sc_np, dtype=torch.long, device=device)
    sn_t = torch.tensor(sn_np, dtype=torch.float32, device=device)
    tf_t = torch.tensor(tf_np, dtype=torch.float32, device=device)
    nt_t = torch.tensor(nt_np, dtype=torch.float32, device=device)
    nt_timesteps_t = torch.tensor(nt_timesteps_np, dtype=torch.float32, device=device)
    nt_mask_t = torch.tensor(nt_mask_np, dtype=torch.float32, device=device)
    
    return sc_t, sn_t, tf_t, nt_t, nt_timesteps_t, nt_mask_t

def collect_background_samples(train_dataloader, num_background=100):
    """
    Collect background samples with consistent dimensionality.
    """
    background_feats = []
    count = 0
    expected_size = get_expected_feature_size()
    
    print(f"Expected feature vector size: {expected_size}")
    
    for batch in train_dataloader:
        B = batch['static_categorical_features'].size(0)
        for i in range(B):
            if count >= num_background:
                break
                
            try:
                fv = flatten_sample_only_feats(
                    batch['static_categorical_features'][i:i+1],
                    batch['static_numerical_features'][i:i+1],
                    batch['ts_features'][i:i+1],
                    batch['notes_embeddings'][i:i+1],
                    batch['notes_timesteps'][i:i+1],
                    batch['notes_mask'][i:i+1]
                )
                
                # Double-check size
                assert len(fv) == expected_size, \
                    f"Unexpected feature vector size: {len(fv)} vs {expected_size}"
                
                background_feats.append(fv)
                count += 1
                
                if count % 10 == 0:
                    print(f"Collected {count}/{num_background} background samples")
                    
            except Exception as e:
                print(f"Warning: Skipping sample due to error: {e}")
            
        if count >= num_background:
            break
            
    if not background_feats:
        raise ValueError("No valid background samples collected")
        
    return np.vstack(background_feats)

@torch.no_grad()
def pipeline_predict_flat(flat_data):
    """
    SHAP-compatible prediction function that handles flattened input data.
    """
    B = flat_data.shape[0]
    sc_list, sn_list, tf_list = [], [], []
    nt_list, nt_timesteps_list, nt_mask_list = [], [], []
    
    for i in range(B):
        sc_i, sn_i, tf_i, nt_i, nt_timesteps_i, nt_mask_i = unflatten_sample_only_feats(flat_data[i])
        sc_list.append(sc_i)
        sn_list.append(sn_i)
        tf_list.append(tf_i)
        nt_list.append(nt_i)
        nt_timesteps_list.append(nt_timesteps_i)
        nt_mask_list.append(nt_mask_i)
    
    # Concatenate batches
    sc_batch = torch.cat(sc_list, dim=0)
    sn_batch = torch.cat(sn_list, dim=0)
    tf_batch = torch.cat(tf_list, dim=0)
    nt_batch = torch.cat(nt_list, dim=0)
    nt_timesteps_batch = torch.cat(nt_timesteps_list, dim=0)
    nt_mask_batch = torch.cat(nt_mask_list, dim=0)
    
    # Create fixed timesteps and mask for time series
    fixed_tm = torch.arange(T_eff, device=device).unsqueeze(0).repeat(B, 1)
    fixed_mk = torch.ones((B, T_eff), device=device, dtype=torch.bool)
    
    return pipeline_predict(
        sc_batch, sn_batch, tf_batch, fixed_tm, fixed_mk,
        nt_batch, nt_timesteps_batch, nt_mask_batch
    )


def compute_shap_for_sample(batch_idx, batch, background_feats, link='identity'):
    """
    Compute SHAP values for a single sample with aggregated notes feature.
    """
    # Extract single sample from batch
    sc = batch['static_categorical_features'][batch_idx:batch_idx+1]
    sn = batch['static_numerical_features'][batch_idx:batch_idx+1]
    tf = batch['ts_features'][batch_idx:batch_idx+1]
    nt = batch['notes_embeddings'][batch_idx:batch_idx+1]
    nt_timesteps = batch['notes_timesteps'][batch_idx:batch_idx+1]
    nt_mask = batch['notes_mask'][batch_idx:batch_idx+1]
    
    # Flatten test sample
    test_feats = flatten_sample_only_feats(
        sc, sn, tf, nt, nt_timesteps, nt_mask
    ).reshape(1, -1)
    
    # Create SHAP explainer with increased nsamples
    explainer = shap.KernelExplainer(
        model=pipeline_predict_flat,
        data=background_feats,
        link=link
    )
    
    # Compute SHAP values
    shap_values = explainer.shap_values(test_feats, nsamples=500)
    
    # Aggregate SHAP values
    offset_ts = num_cat + num_num
    
    def sum_ts_feature(vals, feature_index):
        idxs = [offset_ts + t * F + feature_index for t in range(T_eff)]
        return vals[:, idxs].sum(axis=1)
    
    shap_agg = np.zeros((1, num_cat + num_num + F + 1))  # +1 for aggregated notes
    
    # Aggregate categorical features
    for c in range(num_cat):
        shap_agg[:, c] = shap_values[:, c]
    
    # Aggregate numerical features
    for n in range(num_num):
        shap_agg[:, num_cat + n] = shap_values[:, num_cat + n]
    
    # Aggregate time series features
    for fidx in range(F):
        shap_agg[:, num_cat + num_num + fidx] = sum_ts_feature(shap_values, fidx)
    
    # Notes features (already aggregated)
    shap_agg[:, -1:] = shap_values[:, -1:]
    
    return shap_agg[0]

def run_shap_analysis(test_dataloader, background_feats, max_samples=50):
    """
    Run SHAP analysis with aggregated notes features.
    """
    all_shap_agg = []
    sample_count = 0
    
    for batch in test_dataloader:
        B = batch['static_categorical_features'].size(0)
        for i in range(B):
            if sample_count >= max_samples:
                break

            shap_agg_i = compute_shap_for_sample(i, batch, background_feats)
            all_shap_agg.append(shap_agg_i)
            sample_count += 1
            print(f"Processed sample {sample_count}/{max_samples}")
            
        if sample_count >= max_samples:
            break
            
    return np.vstack(all_shap_agg)

# Main analysis code
# First print the expected feature size
expected_size = get_expected_feature_size()
print(f"Expected feature vector size: {expected_size}")

# Collect background samples
background_feats = collect_background_samples(train_dataloader, num_background=count_bg)
print(f"Successfully collected {len(background_feats)} background samples")
print(f"Feature vector size: {background_feats.shape[1]}")

# Run SHAP analysis
shap_values = run_shap_analysis(train_dataloader, background_feats, max_samples=max_samples)

# Calculate mean absolute SHAP values
mean_abs_shap = np.mean(np.abs(shap_values), axis=0)

# Create feature names
feature_names = (
    CONFIG['static_categorical_cols'] +
    CONFIG['static_numerical_cols'] +
    CONFIG['ts_features'] +
    ['notes_embedding_mean']
)

# Plot results
plt.figure(figsize=(12, 8))
sorted_idx = np.argsort(mean_abs_shap)[::-1]
plt.barh(range(len(feature_names)), mean_abs_shap[sorted_idx][::-1])
plt.yticks(range(len(feature_names)), [feature_names[i] for i in sorted_idx][::-1])
plt.xlabel("Mean |SHAP value|")
plt.title("Feature Importance (with aggregated notes)")
plt.tight_layout()
plt.show()

In [ ]:
def analyze_temporal_similarity(model, dataloader, deltas=[1, 100], cosine_threshold=0.95, num_samples=1000):
    """
    Analyze embedding similarity for different time deltas.
    """
    model.eval()
    
    # Dictionary to store results for each delta
    results = {delta: {
        'cosine_similarities': [],
        'euclidean_distances': [],
        'time_differences': []
    } for delta in deltas}
    
    with torch.no_grad():
        for batch in dataloader:
            # Get batch data
            ts_features = batch['ts_features'].to(device)
            timesteps = batch['timesteps'].to(device)
            mask = batch['mask'].to(device)
            cat_features = batch['static_categorical_features'].to(device)
            num_features = batch['static_numerical_features'].to(device)
            notes_embeddings = batch['notes_embeddings'].to(device)
            notes_timesteps = batch['notes_timesteps'].to(device)
            notes_mask = batch['notes_mask'].to(device)

            # Compute elapsed times
            delta_times = timesteps[:, 1:] - timesteps[:, :-1]
            elapsed_times = torch.cat([delta_times, torch.zeros((delta_times.size(0), 1), device=device)], dim=1)

            # Get model embeddings for all timesteps
            _, lstm_out, _, _ = model(
                x=ts_features,
                elapsed_times=elapsed_times,
                timesteps=timesteps,
                static_features=(cat_features, num_features),
                mask=mask,
                notes_embeddings=notes_embeddings,
                notes_timesteps=notes_timesteps,
                notes_mask=notes_mask
            )
            
            # For each sequence in the batch
            for seq_idx in range(lstm_out.size(0)):
                seq_mask = mask[seq_idx]
                seq_times = timesteps[seq_idx][seq_mask]
                seq_embeddings = lstm_out[seq_idx][seq_mask]
                
                # For each target delta
                for delta in deltas:
                    # Find pairs of timesteps with approximately the target delta
                    for t1 in range(len(seq_times) - 1):
                        for t2 in range(t1 + 1, len(seq_times)):
                            time_diff = (seq_times[t2] - seq_times[t1]).item()
                            
                            # Check if this pair's time difference is close to target delta
                            # Allow for some flexibility (±10% of delta)
                            if 0.9 * delta <= time_diff <= 1.1 * delta:
                                emb1 = seq_embeddings[t1]
                                emb2 = seq_embeddings[t2]
                                
                                # Compute similarities
                                cosine_sim = torch_F.cosine_similarity(emb1.unsqueeze(0), 
                                                               emb2.unsqueeze(0)).item()
                                euclidean_dist = torch.norm(emb1 - emb2).item()
                                
                                # Store results
                                results[delta]['cosine_similarities'].append(cosine_sim)
                                results[delta]['euclidean_distances'].append(euclidean_dist)
                                results[delta]['time_differences'].append(time_diff)
                                
                                if len(results[delta]['cosine_similarities']) >= num_samples:
                                    break
                            
                        if len(results[delta]['cosine_similarities']) >= num_samples:
                            break
                    
                    if len(results[delta]['cosine_similarities']) >= num_samples:
                        break
            
            # Check if we have enough samples for all deltas
            if all(len(results[d]['cosine_similarities']) >= num_samples for d in deltas):
                break
    
    # Print analysis
    print("\n=== Temporal Similarity Analysis ===")
    for delta in deltas:
        cosine_sims = np.array(results[delta]['cosine_similarities'])
        euclidean_dists = np.array(results[delta]['euclidean_distances'])
        time_diffs = np.array(results[delta]['time_differences'])
        
        print(f"\nResults for delta ≈ {delta}:")
        print(f"Number of pairs analyzed: {len(cosine_sims)}")
        print(f"Average cosine similarity: {np.mean(cosine_sims):.4f} ± {np.std(cosine_sims):.4f}")
        print(f"Average euclidean distance: {np.mean(euclidean_dists):.4f} ± {np.std(euclidean_dists):.4f}")
        print(f"Proportion of highly similar embeddings (cosine > {cosine_threshold}): "
              f"{np.mean(cosine_sims > cosine_threshold):.4f}")
        
        # Additional statistics
        percentiles = np.percentile(cosine_sims, [25, 50, 75])
        print(f"Cosine similarity quartiles: {percentiles[0]:.4f}, {percentiles[1]:.4f}, {percentiles[2]:.4f}")
    
    # Return results for plotting
    return {str(delta): {
        'cosine_similarities': results[delta]['cosine_similarities'],
        'euclidean_distances': results[delta]['euclidean_distances'],
        'time_differences': results[delta]['time_differences']
    } for delta in deltas}

# Example usage:
results = analyze_temporal_similarity(
    model, 
    test_dataloader,
    deltas=[1, 100],
    cosine_threshold=0.95,
    num_samples=1000
)
results = analyze_temporal_similarity(model, test_dataloader)

In [ ]:
# Compute perturbation sensitivity analysis
# This helps us see how changes in input features affect the latent dimensions
def compute_perturbation_sensitivity(model, dataloader, num_samples=300, epsilon=1e-4, threshold=1e-4):
    """
    Measure how perturbations in input features affect latent dimensions.
    
    Args:
        model: trained model
        dataloader: dataloader with samples
        num_samples: number of samples to analyze
        epsilon: size of perturbation noise
        threshold: minimum change to consider a dimension affected
    
    Returns:
        Dictionary with results per feature
    """
    perturbation_results = {}
    all_feature_names = CONFIG['static_categorical_cols'] + CONFIG['static_numerical_cols'] + CONFIG['ts_features']
    
    print(f"Running perturbation sensitivity analysis with threshold={threshold}")
    
    # Initialize results structure
    for feature in all_feature_names:
        perturbation_results[feature] = {
            'affected_dims': 0,
            'dims_changed': []
        }
    
    model.eval()
    sample_count = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Processing batches"):
            # Skip if we have enough samples
            if sample_count >= num_samples:
                break
                
            # Get batch data
            ts_features = batch['ts_features'].to(device)
            timesteps = batch['timesteps'].to(device)
            mask = batch['mask'].to(device)
            cat_features = batch['static_categorical_features'].to(device)
            num_features = batch['static_numerical_features'].to(device)
            notes_embeddings = batch['notes_embeddings'].to(device)
            notes_timesteps = batch['notes_timesteps'].to(device)
            notes_mask = batch['notes_mask'].to(device)
            
            # Compute intervals for temporal model
            delta_times = timesteps[:, 1:] - timesteps[:, :-1]
            elapsed_times = torch.cat([delta_times, torch.zeros((delta_times.size(0), 1), device=device)], dim=1)
            
            # Get baseline representations with unmodified inputs
            _, baseline_repr, _, _ = model(
                x=ts_features,
                elapsed_times=elapsed_times,
                timesteps=timesteps,
                static_features=(cat_features, num_features),
                mask=mask,
                notes_embeddings=notes_embeddings,
                notes_timesteps=notes_timesteps,
                notes_mask=notes_mask
            )
            # Get final timestep representation
            baseline_repr = baseline_repr[:, -1, :]  # (B, H)
            
            # Run 50 trials for each sample to get stable estimates
            n_trials = 50
            
            # Process each sample in batch
            for b in range(min(ts_features.size(0), num_samples - sample_count)):
                # For each feature, perturb and measure change
                for feature_idx, feature_name in enumerate(all_feature_names):
                    # For static categorical features
                    if feature_idx < len(CONFIG['static_categorical_cols']):
                        # Skip categorical features (would need one-hot encoding)
                        continue
                    
                    # For static numerical features
                    elif feature_idx < len(CONFIG['static_categorical_cols']) + len(CONFIG['static_numerical_cols']):
                        num_idx = feature_idx - len(CONFIG['static_categorical_cols'])
                        for trial in range(n_trials):
                            # Create copy of original features
                            perturbed_num = num_features[b:b+1].clone()
                            # Add noise to specific feature
                            noise = torch.randn(1) * epsilon
                            perturbed_num[0, num_idx] += noise
                            
                            # Forward pass with perturbed input
                            _, perturbed_repr, _, _ = model(
                                x=ts_features[b:b+1],
                                elapsed_times=elapsed_times[b:b+1],
                                timesteps=timesteps[b:b+1],
                                static_features=(cat_features[b:b+1], perturbed_num),
                                mask=mask[b:b+1],
                                notes_embeddings=notes_embeddings[b:b+1],
                                notes_timesteps=notes_timesteps[b:b+1], 
                                notes_mask=notes_mask[b:b+1]
                            )
                            
                            # Measure which dimensions changed
                            diff = (perturbed_repr[:, -1, :] - baseline_repr[b:b+1]).abs()
                            changed_dims = (diff > threshold).sum().item()
                            
                            # Record which specific dimensions changed
                            dims_changed = torch.where(diff[0] > threshold)[0].cpu().numpy().tolist()
                            perturbation_results[feature_name]['dims_changed'].extend(dims_changed)
                            
                            # Update average affected dimensions
                            if 'affected_dims_list' not in perturbation_results[feature_name]:
                                perturbation_results[feature_name]['affected_dims_list'] = []
                            perturbation_results[feature_name]['affected_dims_list'].append(changed_dims)
                    
                    # For time series features
                    else:
                        ts_idx = feature_idx - (len(CONFIG['static_categorical_cols']) + len(CONFIG['static_numerical_cols']))
                        for trial in range(n_trials):
                            # Create copy of original features
                            perturbed_ts = ts_features[b:b+1].clone()
                            # Add noise to specific feature across all timesteps where mask is true
                            for t in range(ts_features.size(1)):
                                if mask[b, t]:
                                    noise = torch.randn(1) * epsilon
                                    perturbed_ts[0, t, ts_idx] += noise
                            
                            # Forward pass with perturbed input
                            _, perturbed_repr, _, _ = model(
                                x=perturbed_ts,
                                elapsed_times=elapsed_times[b:b+1],
                                timesteps=timesteps[b:b+1],
                                static_features=(cat_features[b:b+1], num_features[b:b+1]),
                                mask=mask[b:b+1],
                                notes_embeddings=notes_embeddings[b:b+1],
                                notes_timesteps=notes_timesteps[b:b+1],
                                notes_mask=notes_mask[b:b+1]
                            )
                            
                            # Measure which dimensions changed
                            diff = (perturbed_repr[:, -1, :] - baseline_repr[b:b+1]).abs()
                            changed_dims = (diff > threshold).sum().item()
                            
                            # Record which specific dimensions changed
                            dims_changed = torch.where(diff[0] > threshold)[0].cpu().numpy().tolist()
                            perturbation_results[feature_name]['dims_changed'].extend(dims_changed)
                            
                            # Update average affected dimensions
                            if 'affected_dims_list' not in perturbation_results[feature_name]:
                                perturbation_results[feature_name]['affected_dims_list'] = []
                            perturbation_results[feature_name]['affected_dims_list'].append(changed_dims)
                
                # Increment sample counter
                sample_count += 1
                if sample_count % 10 == 0:
                    print(f"Processed {sample_count}/{num_samples} samples")
                    
                if sample_count >= num_samples:
                    break
    
    # Calculate average affected dimensions for each feature
    for feature in all_feature_names:
        if 'affected_dims_list' in perturbation_results[feature]:
            perturbation_results[feature]['affected_dims'] = np.mean(perturbation_results[feature]['affected_dims_list'])
            
            # Get unique dimensions affected
            unique_dims = np.unique(perturbation_results[feature]['dims_changed'])
            perturbation_results[feature]['unique_dims'] = len(unique_dims)
            
            # Calculate concentration (fewer affected dimensions = higher concentration)
            latent_dim = baseline_repr.size(1)
            perturbation_results[feature]['concentration'] = 1.0 - (perturbation_results[feature]['unique_dims'] / latent_dim)
            
    # Print results
    print("\n=== Perturbation Sensitivity Results ===")
    print(f"{'Feature':<25} {'Affected Dims':<15} {'Concentration':<15}")
    print("-" * 55)
    
    # Sort by number of affected dimensions
    sorted_features = sorted(
        [(f, perturbation_results[f].get('affected_dims', 0)) for f in all_feature_names 
         if 'affected_dims' in perturbation_results[f]],
        key=lambda x: x[1],
        reverse=True
    )
    
    for feature, _ in sorted_features:
        result = perturbation_results[feature]
        if 'affected_dims' in result:
            print(f"{feature:<25} {result['affected_dims']:<15.2f} {result.get('concentration', 0):<15.4f}")
    
    return perturbation_results

# Run perturbation analysis 
perturbation_results = compute_perturbation_sensitivity(model, test_dataloader, num_samples=100)

In [ ]:
# Compute disentanglement, completeness, and informativeness using DCI framework
class DCI_Metrics:
    """Implementation of DCI metrics from 'A Framework for the Quantitative Evaluation of Disentangled Representations'"""
    
    def __init__(self, model, dataloader, device):
        self.model = model
        self.dataloader = dataloader
        self.device = device
        self.representation_dim = 512  # Assuming latent dim is 512
        
    def extract_representations_and_factors(self, num_samples=500):
        """Extract latent representations and ground truth factors for analysis"""
        representations = []
        factors_static = []
        factors_ts = []
        
        self.model.eval()
        with torch.no_grad():
            for batch in tqdm(self.dataloader, desc="Extracting representations"):
                # Get batch data
                ts_features = batch['ts_features'].to(self.device)
                timesteps = batch['timesteps'].to(self.device)
                mask = batch['mask'].to(self.device)
                cat_features = batch['static_categorical_features'].to(self.device)
                num_features = batch['static_numerical_features'].to(self.device)
                
                notes_embeddings = batch['notes_embeddings'].to(self.device)
                notes_timesteps = batch['notes_timesteps'].to(self.device)
                notes_mask = batch['notes_mask'].to(self.device)
                
                # Compute intervals for temporal model
                delta_times = timesteps[:, 1:] - timesteps[:, :-1]
                elapsed_times = torch.cat([delta_times, torch.zeros((delta_times.size(0), 1), device=self.device)], dim=1)
                
                # Get representations
                _, lstm_out, _, _ = self.model(
                    x=ts_features,
                    elapsed_times=elapsed_times,
                    timesteps=timesteps,
                    static_features=(cat_features, num_features),
                    mask=mask,
                    notes_embeddings=notes_embeddings,
                    notes_timesteps=notes_timesteps,
                    notes_mask=notes_mask
                )
                
                # Use final timestep representation
                batch_repr = lstm_out[:, -1, :].cpu().numpy()
                
                # Get ground truth factors
                batch_static = np.column_stack([
                    cat_features.cpu().numpy(),
                    num_features.cpu().numpy()
                ])
                
                # Get last timestep of time series as factors
                batch_ts = ts_features[:, -1, :].cpu().numpy()
                
                representations.append(batch_repr)
                factors_static.append(batch_static)
                factors_ts.append(batch_ts)
                
                if len(representations) * batch_repr.shape[0] >= num_samples:
                    break
        
        self.representations = np.vstack(representations)[:num_samples]
        self.factors_static = np.vstack(factors_static)[:num_samples]
        self.factors_ts = np.vstack(factors_ts)[:num_samples]
        
        # Combine factors for overall evaluation
        self.factors_all = np.hstack([self.factors_static, self.factors_ts])
        
        # Get feature names
        self.static_features = CONFIG['static_categorical_cols'] + CONFIG['static_numerical_cols']
        self.ts_features = CONFIG['ts_features']
        self.all_features = self.static_features + self.ts_features
        
        print(f"Extracted representations shape: {self.representations.shape}")
        print(f"Extracted factors shape: {self.factors_all.shape}")
        
    def compute_importance_matrix(self, factors):
        """Compute importance scores between latent dimensions and factors"""
        from sklearn.ensemble import RandomForestRegressor
        
        n_factors = factors.shape[1]
        importance_matrix = np.zeros((n_factors, self.representation_dim))
        
        for f_idx in tqdm(range(n_factors), desc="Computing importance matrix"):
            # Train a random forest to predict the factor from latent dimensions
            rf = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)
            rf.fit(self.representations, factors[:, f_idx])
            
            # Get feature importances
            importance_matrix[f_idx] = rf.feature_importances_
            
        return importance_matrix
        
    def compute_disentanglement(self, importance_matrix):
        """Compute disentanglement score"""
        # For each latent dimension, compute entropy of importance distribution
        n_factors = importance_matrix.shape[0]
        per_dim_entropy = []
        
        for dim in range(self.representation_dim):
            # Get importance of this dimension for each factor
            dim_importance = importance_matrix[:, dim]
            
            # Normalize to sum to 1
            if np.sum(dim_importance) > 0:
                dim_importance = dim_importance / np.sum(dim_importance)
                
                # Compute entropy
                entropy = -np.sum(dim_importance * np.log(dim_importance + 1e-10)) / np.log(n_factors)
                
                # Disentanglement is 1 - entropy (lower entropy means higher disentanglement)
                disentanglement = 1 - entropy
                per_dim_entropy.append(disentanglement)
        
        # Overall score is weighted average based on relative importance
        dim_importance_weight = np.sum(importance_matrix, axis=0)
        dim_importance_weight = dim_importance_weight / np.sum(dim_importance_weight)
        
        # For dimensions with non-zero importance
        non_zero_dims = np.where(dim_importance_weight > 0)[0]
        disentanglement_score = np.sum(
            [per_dim_entropy[i] * dim_importance_weight[non_zero_dims[i]] 
             for i in range(len(non_zero_dims))]
        )
        
        return disentanglement_score
        
    def compute_completeness(self, importance_matrix):
        """Compute completeness score"""
        # For each factor, compute entropy of importance distribution across dimensions
        n_factors = importance_matrix.shape[0]
        per_factor_completeness = []
        
        for f in range(n_factors):
            # Get importance of each dimension for this factor
            factor_importance = importance_matrix[f, :]
            
            # Normalize to sum to 1
            if np.sum(factor_importance) > 0:
                factor_importance = factor_importance / np.sum(factor_importance)
                
                # Compute entropy
                entropy = -np.sum(factor_importance * np.log(factor_importance + 1e-10)) / np.log(self.representation_dim)
                
                # Completeness is 1 - entropy
                completeness = 1 - entropy
                per_factor_completeness.append(completeness)
        
        # Overall score is average across factors
        completeness_score = np.mean(per_factor_completeness)
        
        return completeness_score
        
    def compute_informativeness(self, factors):
        """Compute informativeness score"""
        from sklearn.ensemble import RandomForestRegressor
        from sklearn.model_selection import train_test_split
        
        # Split data for evaluation
        X_train, X_test, y_train, y_test = train_test_split(
            self.representations, factors, test_size=0.2, random_state=42
        )
        
        # Normalize targets for consistent scale
        y_mean = np.mean(y_train, axis=0)
        y_std = np.std(y_train, axis=0)
        y_std[y_std == 0] = 1  # Prevent division by zero
        
        y_train_norm = (y_train - y_mean) / y_std
        y_test_norm = (y_test - y_mean) / y_std
        
        # Train a model to predict all factors simultaneously
        n_factors = factors.shape[1]
        r2_scores = []
        
        for f in range(n_factors):
            rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
            rf.fit(X_train, y_train_norm[:, f])
            
            # Get R^2 score
            preds = rf.predict(X_test)
            r2 = max(0, 1 - np.sum((y_test_norm[:, f] - preds)**2) / np.sum((y_test_norm[:, f] - np.mean(y_test_norm[:, f]))**2))
            r2_scores.append(r2)
        
        # Average R^2 across factors
        informativeness = np.mean(r2_scores)
        
        return informativeness
        
    def compute_all_metrics(self):
        """Compute all DCI metrics"""
        # Extract data if not already done
        if not hasattr(self, 'representations'):
            self.extract_representations_and_factors()
        
        # Compute importance matrices
        self.importance_matrix_static = self.compute_importance_matrix(self.factors_static)
        self.importance_matrix_ts = self.compute_importance_matrix(self.factors_ts)
        self.importance_matrix_all = self.compute_importance_matrix(self.factors_all)
        
        # Compute metrics for static features
        disentanglement_static = self.compute_disentanglement(self.importance_matrix_static)
        completeness_static = self.compute_completeness(self.importance_matrix_static)
        informativeness_static = self.compute_informativeness(self.factors_static)
        
        # Compute metrics for time series features
        disentanglement_ts = self.compute_disentanglement(self.importance_matrix_ts)
        completeness_ts = self.compute_completeness(self.importance_matrix_ts)
        informativeness_ts = self.compute_informativeness(self.factors_ts)
        
        # Compute metrics for all features
        disentanglement_all = self.compute_disentanglement(self.importance_matrix_all)
        completeness_all = self.compute_completeness(self.importance_matrix_all)
        informativeness_all = self.compute_informativeness(self.factors_all)
        
        # Compile results
        results = {
            'static_features': {
                'disentanglement': disentanglement_static,
                'completeness': completeness_static,
                'informativeness': informativeness_static
            },
            'ts_features': {
                'disentanglement': disentanglement_ts,
                'completeness': completeness_ts,
                'informativeness': informativeness_ts
            },
            'all_features': {
                'disentanglement': disentanglement_all,
                'completeness': completeness_all,
                'informativeness': informativeness_all
            }
        }
        
        return results

# Calculate DCI metrics for our model
dci = DCI_Metrics(model, test_dataloader, device)
metrics = dci.compute_all_metrics()

# Print results
print("\n=== DCI Metrics ===")
for feature_set in metrics:
    print(f"\n{feature_set.upper()}:")
    print(f"Disentanglement: {metrics[feature_set]['disentanglement']:.4f}")
    print(f"Completeness: {metrics[feature_set]['completeness']:.4f}")
    print(f"Informativeness: {metrics[feature_set]['informativeness']:.4f}")

# aslo compare metrics across our three model variants
def compute_metrics_for_model(model_path, dataloader, device):
    """Load model and compute metrics"""
    model.load_state_dict(torch.load(model_path, weights_only=True))
    model.eval()
    model.to(device)
    
    dci = DCI_Metrics(model, dataloader, device)
    return dci.compute_all_metrics()['all_features']

# Compute metrics for three model variants with different loss functions
recon_only_model = '../models/recon_only.pt'
recon_decorr_model = '../models/recon_decorr.pt'
full_model = '../models/complete10ep_cont4.pt'

print("\n=== Comparison Across Model Variants ===")
print(f"{'Model':<20} {'Disentanglement':<15} {'Completeness':<15} {'Informativeness':<15}")
print("-" * 65)

recon_metrics = compute_metrics_for_model(recon_only_model, test_dataloader, device)
print(f"{'$\\mathcal{L}_{recon}$':<20} {recon_metrics['disentanglement']:.2f} {recon_metrics['completeness']:.2f} {recon_metrics['informativeness']:.2f}")

decorr_metrics = compute_metrics_for_model(recon_decorr_model, test_dataloader, device)
print(f"{'+ $\\mathcal{L}_{decorr}$':<20} {decorr_metrics['disentanglement']:.2f} {decorr_metrics['completeness']:.2f} {decorr_metrics['informativeness']:.2f}")

full_metrics = compute_metrics_for_model(full_model, test_dataloader, device)
print(f"{'+ $\\mathcal{L}_{disent}$':<20} {full_metrics['disentanglement']:.2f} {full_metrics['completeness']:.2f} {full_metrics['informativeness']:.2f}")

# Export results to JSON for table generation
with open('../data/results/dci_metrics.json', 'w') as f:
    json.dump({
        '$\\mathcal{L}_{recon}$': recon_metrics,
        '+ $\\mathcal{L}_{decorr}$': decorr_metrics,
        '+ $\\mathcal{L}_{disent}$': full_metrics
    }, f)